In [399]:
import numpy as np

In [400]:
board = np.full((3,3), " ")

In [401]:
def idx_to_row_colum(idx):
    row = idx // 3
    colum = idx % 3
    return row, colum

In [402]:
def play_to_board(board:np.ndarray, idx:int, value:str):
    x = board.copy()
    x[idx_to_row_colum(idx)] = value
    return x

In [403]:
def leagal_move_indicies(board:np.ndarray):
    return np.where((board.flatten()==" "))[0]

In [404]:
def is_game_over(board):
    lines = []

    # rows and columns
    lines.extend(board)
    lines.extend(board.T)

    # diagonals
    lines.append(np.diag(board))
    lines.append(np.diag(np.fliplr(board)))

    for line in lines:
        values = set(line)
        if values == {"X"}:
            return True, 1
        if values == {"O"}:
            return True, -1

    if len(leagal_move_indicies(board)) == 0:
        return True, 0

    return False, None

In [405]:
x_win_board = np.array([
    ["X", "O", "X"],
    ["O", "X", " "],
    ["X", " ", "O"],
])

unfinished_board = np.array([
    ["X", "O", "X"],
    ["O", "X", " "],
    [" ", " ", "O"],
])

about_to_win_board = np.array([
    ["X", " ", " "],
    ["O", "X", " "],
    ["X", "O", " "],
])

print(is_game_over(x_win_board))
print(is_game_over(unfinished_board))

(True, 1)
(False, None)


In [406]:
board = play_to_board(board, 4, "X")
print(leagal_move_indicies(board))

[0 1 2 3 5 6 7 8]


In [407]:
def minimax(depth, board, maximizing_player):
    res = is_game_over(board)
    if res[0]:
        return res[1]
    if depth == 0:
        return 0

    if maximizing_player:
        max_eval = -float("inf")
        moves = leagal_move_indicies(board)
        for move in moves:
            eval = minimax(depth-1, play_to_board(board, move, "X"), not maximizing_player)
            max_eval = max(eval, max_eval)
        return max_eval
    
    max_eval = float("inf")
    moves = leagal_move_indicies(board)
    for move in moves:
        eval = minimax(depth-1, play_to_board(board, move, "O"), not maximizing_player)
        max_eval = min(eval, max_eval)
    return max_eval

In [408]:
minimax(9, about_to_win_board, False)

1

In [409]:
test_board = np.array([
    ["X", "O", "X"],
    ["O", "X", " "],
    [" ", " ", "O"],
])

In [410]:
def telemetry_moves(board, turn="O", depth=30):
    move_scores = []
    for move in leagal_move_indicies(board):
        new_board = play_to_board(board, move, turn)
        next_player_is_x = turn == "O"
        score = minimax(depth=depth - 1, board=new_board, maximizing_player=next_player_is_x)
        move_scores.append({
            "move": int(move),
            "player": turn,
            "score_for_X": score,
        })

    return move_scores

In [411]:
scores = telemetry_moves(board, turn="O", depth=9)

print("O candidate move scores:")
for item in scores:
    print(f"move {item['move']} -> score_for_X {item['score_for_X']}")

best_for_o = min(scores, key=lambda item: item["score_for_X"])
print(f"Chosen O move: {best_for_o['move']} with score_for_X {best_for_o['score_for_X']}")

if all(item["score_for_X"] == 1 for item in scores):
    print("All O moves still score 1, so X can force a win no matter what O chooses.")

O candidate move scores:
move 0 -> score_for_X 0
move 1 -> score_for_X 1
move 2 -> score_for_X 0
move 3 -> score_for_X 1
move 5 -> score_for_X 1
move 6 -> score_for_X 0
move 7 -> score_for_X 1
move 8 -> score_for_X 0
Chosen O move: 0 with score_for_X 0
